# 📊 Первичный анализ

## 1. Импорт библиотек и загрузка сырых данных

In [1]:
import os
os.environ["PYTHONDONTWRITEBYTECODE"] = "0"

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, Markdown

from spam_utils import load_raw_data, clean_spam_data
df_raw = load_raw_data()
print(f'Размер сырого датасета: {df_raw.shape}')
display(df_raw.head())

Размер сырого датасета: (5572, 5)


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


## 2. Поиск дубликатов и очистка

In [2]:
duplicates_count = df_raw.duplicated().sum()
print(f"🔍 Найдено полных дубликатов строк: {duplicates_count}")

# Очищаем данные для дальнейшего анализа
df = clean_spam_data(df_raw)
print(f"Размер исходного датасета: {df_raw.shape}")
print(f"Размер очищенного датасета: {df.shape}")
display(df.head(3))

🔍 Найдено полных дубликатов строк: 403
Размер исходного датасета: (5572, 5)
Размер очищенного датасета: (5169, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...


## 3. Проверка на пустые сообщения и пропуски (NaN)

In [3]:
total_messages = len(df)

# Проверка на NaN и на пустые строки (или строки из пробелов)
nan_count = df['message'].isnull().sum()
empty_strings = (df['message'].str.strip() == '').sum()
empty_messages = nan_count + empty_strings

print(f"📝 Всего сообщений (после удаления дублей): {total_messages}")
print(f"🕳️ Пустых сообщений: {empty_messages}")
print(f"❌ Пропусков (NaN) в целевой колонке 'message_raw': {nan_count}")

📝 Всего сообщений (после удаления дублей): 5169
🕳️ Пустых сообщений: 0
❌ Пропусков (NaN) в целевой колонке 'message_raw': 0


## 4. Распределение классов и сбалансированность

In [4]:
class_counts = df['label'].value_counts().reset_index()
class_counts.columns = ['Класс', 'Количество']
class_counts['Доля'] = (class_counts['Количество'] / total_messages * 100).round(2).astype(str) + '%'

display(Markdown("### Таблица распределения классов"))
display(class_counts)


### Таблица распределения классов

,Класс,Количество,Доля
0,ham,4516,87.37%
1,spam,653,12.63%


## 5. Примеры Spam и Ham сообщений

In [5]:
display(Markdown("### 🟢 Примеры HAM (Легальные сообщения)"))
display(df[df['label'] == 'ham']['message'].sample(3, random_state=424).to_frame())

display(Markdown("### 🔴 Примеры SPAM (Спам)"))
display(df[df['label'] == 'spam']['message'].sample(3, random_state=424).to_frame())

### 🟢 Примеры HAM (Легальные сообщения)

,message
1263,Those cocksuckers. If it makes you feel better...
1578,Sounds like you have many talents! would you l...
4215,Not able to do anything.


### 🔴 Примеры SPAM (Спам)

,message
2319,SMS SERVICES For your inclusive text credits p...
3705,"Free Msg: get Gnarls Barkleys \Crazy\"" rington..."
4125,For your chance to WIN a FREE Bluetooth Headse...


## 6. Простой анализ длины сообщений

In [6]:
df['msg_length'] = df['message'].apply(len)

fig_hist = px.histogram(
    df, 
    x='msg_length', 
    color='label',
    barmode='overlay',
    marginal='box',
    title='Распределение длины сообщений (Spam vs Ham)',
    color_discrete_map={'ham': '#636EFA', 'spam': '#EF553B'},
    opacity=0.7
)
fig_hist.update_layout(xaxis_title='Длина сообщения (символы)', yaxis_title='Частота')
fig_hist.show()